# GFZ-Verteilung Österreich – reproduzierbare Bundeslandhochrechnung

2021er Bestandsmodell. **Wien → Wien; Salzburg → alle anderen Bundesländer.** Ausnahmen werden nur bei `fallback="vienna"` ausdrücklich ausgewiesen. Berechnungslogik in `gfz_hydrator.py` und nicht in Notebook-Zellen duplizieren.


In [ ]:
from pathlib import Path
from argparse import Namespace
from gfz_hydrator import read_samples, read_statistics, run
from plot_gfz import make_figures

DATA = Path("data")
BUILDINGS = DATA / "rohdaten_python.xlsx"
STATISTICS = DATA / "GFZ_Analyse_AT.xlsx"
OUT = Path("outputs/recalculated")
FALLBACK = "vienna"   # Alternative "error" -> zeigt nicht abgedeckte Kombinationen


## 1. Datenprüfung

**GFZ** und **BGF** stammen gebäudescharf aus Wien/Salzburg; Statistikdaten nach Bundesland, Geschossklasse und Hauptnutzung. Zur Vollständigkeit der Klassenzuordnungen siehe `reference_coverage.csv`.


In [ ]:
buildings, building_meta = read_samples(BUILDINGS, max_gfz=20.0)
targets, target_meta = read_statistics(STATISTICS, ngf_per_bgf=0.8)
display(buildings.groupby(["Bundesland", "GK"])["BGF"].agg(["count", "sum"]).round(1))
print(building_meta)
print(target_meta)


## 2. Hochrechnung inklusive Flächenbilanz

`--fallback vienna` verwendet bei weniger als fünf Gebäuden eine **gekennzeichnete** Ersatzreferenz. Diese Ersatzregel wurde am 21.10.2026 vom Auftraggeber freigegeben. Die Bestandsrechnung ist noch keine Kalibrierung des PEQ-Standards für 2040.


In [ ]:
args = Namespace(buildings=BUILDINGS, statistics=STATISTICS, out=OUT, ngf_per_bgf=0.8,
                 gfz_step=0.1, gfz_max=20.0, min_samples=5, method="kde", fallback=FALLBACK)
report = run(args)
for k in ("statistics_bgf_total_m2", "allocated_bgf_m2", "total_allocation_residual_m2", "n_fallback_cells", "fallback_share_of_total_bgf"):
    print(f"{k}: {report[k]}")


## 3. Validierung und Figuren

Die Hauptdaten liegen in `gfz_distribution_by_province_use_storeys.csv`. Für die Berichtstexte primär `allocation_audit.csv` und `validation.json` prüfen.


In [ ]:
import pandas as pd
audit = pd.read_csv(OUT / "allocation_audit.csv")
display(audit.loc[audit.status != "reference-exact", ["Bundesland", "GK", "GWR_Nutzung", "status", "reference_region", "reference_gk", "BGF_target_m2"]])
make_figures(OUT, OUT)
from IPython.display import Image, display
display(Image(filename=str(OUT / "gfz_oesterreich_nutzung.png")))
display(Image(filename=str(OUT / "fallback_bundeslaender.png")))


## 4. Methodische Einschränkungen

Phase 2 wird nur dokumentiert, ohne Sensitivitäts- oder Nachbearbeitungsschleife. Offene Punkte siehe `METHODENSTAND.md`. `method="histogram"` bleibt für eine spätere Untersuchung verfügbar.


In [ ]:
# Optionale zweite Rechnung:
# args.method = "histogram"
# args.out = Path("outputs_hist")
# report_hist = run(args)
